# Recruitment Milestones — NANO & NICO Studies (NIH Reporting)

**Projects covered**

| Key | REDCap project | PID | Notes |
|---|---|---|---|
| `NANO` | NANO Study Surveys | 4218 | De-identified export token (all dates stripped, no logging) |
| `NICO` | NICO Study | 3836 | Token supplied labeled *"MICO"*; the project title is **NICO Study** and the phases in the brief called it *"NICU"* — same project. |

This notebook rebuilds the **Recruitment Milestones** table (Total / Racial-Minority /
Hispanic-Ethnicity cumulative recruitment) for each project directly from REDCap, following
an auditable, phase-by-phase workflow:

0. **Secure setup** — tokens are read from the environment first and are *never printed*.
1. **Discovery** — connect, download project structure & data dictionary, and *inspect*
   (not assume) the fields that define recruitment, race, ethnicity, and exclusions.
2. **Recruitment ground truth** — a participant-level classification (include / exclude /
   manual-review) with an explicit reason for every record.
3. **Race & ethnicity classification** — NIH-consistent rules from the data dictionary's
   coded values, with demographic-quality reports.
4. **Milestones & targets** — provenance table; targets that cannot be verified are marked
   *Pending Target Verification* rather than invented.
5. **Cumulative actuals** — unique valid participants per category, per milestone.
6. **Final tables + validation outputs** — styled tables plus the full QA pack.

> **Ratio rule (as specified):** `Actual/Target Ratio = Actual ÷ Current-Target × 100`.
> When the current target is **0 or unknown**, the ratio and status render **"N/A"** — a zero
> target is *never* used as a substitute for an unknown target.
> **Status:** *On Target* (green) when cumulative actual ≥ current cumulative target,
> otherwise *Below Target* (amber/red).

---

### ⚠️ Read before trusting the historical columns
* **NANO** — the token is *de-identified*: REDCap strips **every** date server-side, so
  historical per-milestone columns **cannot** be time-bucketed from live data. The notebook
  computes the **current** cumulative live (this reproduces the published column exactly:
  Total **219**, Racial-minority **99**, Hispanic **16**) and fills earlier columns from the
  study's already-published `HISTORICAL_ACTUALS` seed.
* **NICO** — no verified protocol **enrollment/consent date** field exists, race is captured in
  a non-standard field, and **NIH milestone dates/targets were not supplied**. The notebook
  therefore computes **current** actuals with full data-quality caveats and marks the NIH table
  **"Pending Target Verification / Data Access Required."**


## Phase 0 — Secure configuration

Tokens are pulled from environment variables first (`NANO_API_TOKEN`, `NICO_API_TOKEN`); the
literal values supplied for this engagement are kept only as a fallback so the notebook runs
out-of-the-box. **The token strings are never printed, logged, or exported** — the cell prints
only a masked length confirmation. For anything shared or committed, set the environment
variables (or Colab secrets) and delete the literals below.

In [1]:
# =====================================================================
# Phase 0 — Secure configuration  (edit here; never print tokens)
# =====================================================================
import os, datetime as dt

API_URL = "https://redcap.research.sc.edu/api/"

# Env-var first, literal fallback. Do NOT echo these anywhere.
TOKENS = {
    "NANO": os.environ.get("NANO_API_TOKEN", "6324B3FAA4E18D8D513776801CFABA20"),
    "NICO": os.environ.get("NICO_API_TOKEN", "FFA818F26E9CA0A23DD4F54827D00461"),
}

# Report cut-off / "as of" date for every cumulative count and the current-milestone marker.
REPORT_DATE = dt.date.today()          # or dt.date(2026, 7, 23)

# Output folders: public = aggregate only (safe to share); secure = participant-level audit.
PUBLIC_DIR = "recruitment_outputs"
SECURE_DIR = "recruitment_audit_secure"   # add to .gitignore; internal QA only
os.makedirs(PUBLIC_DIR, exist_ok=True)
os.makedirs(SECURE_DIR, exist_ok=True)

def _mask(tok: str) -> str:
    return f"loaded ({len(tok)} chars)" if tok else "MISSING"

print("Report cut-off date :", REPORT_DATE.isoformat())
for k, v in TOKENS.items():
    print(f"  token[{k}] : {_mask(v)}")     # never prints the value


Report cut-off date : 2026-07-23
  token[NANO] : loaded (32 chars)
  token[NICO] : loaded (32 chars)


### Verified per-project field map

These mappings were established by **inspecting the live data dictionary and field fill-rates**
(Phase 1 re-verifies them at run time and warns if a better-populated field appears). Key
decisions:

* **Race — NANO** uses `fif_childrace` (checkbox, 6 codes). **NICO** uses **`race`** (checkbox on
  *infant_demographics*, populated 82/82) rather than `fif_childrace` (only 18/82). NICO's `race`
  has a different coding and even lists *"Hispanic or Latino"* as a race option (code 4).
* **Racial minority** = any specific non-White race; **White**, **Unknown/Other**, and **missing**
  are *not* counted as minority (missing is never treated as non-minority).
* **Ethnicity — NANO** uses `fif_childethnicity` (1 = Hispanic/Latino). **NICO** ethnicity is
  under-captured; it is taken from `fif_childethnicity`, the PRAPARE self-report `ethnicity`
  (0 = Yes), and race code 4 — unioned and flagged provisional.
* **Exclusions** (both): `demo_ineligible`, `demo_unenrolled`, `demo_exclude`. **NICO** adds
  `dual_enrolled` (0 = NICO only, 1 = also in NANO) for cross-project de-duplication.

In [2]:
# =====================================================================
# Phase 0b — Verified per-project configuration
# =====================================================================
PROJECT_CONFIG = {
    "NANO": {
        "label": "NANO Study",
        "grant": "MH132925",
        "study_title": "The Role of Autonomic Regulation of Attention in the Emergence of ASD",
        "record_id": "demo_id",
        "race_field": "fif_childrace",   # checkbox
        # 1 AI/AN, 2 Asian, 3 NHPI, 4 Black/African American, 5 White, 6 Unknown/Other
        "race_minority_codes": [1, 2, 3, 4],
        "race_white_codes":    [5],
        "race_unknown_codes":  [6],
        "race_hispanic_codes": [],       # race field carries no Hispanic option
        "eth_field": "fif_childethnicity",   # radio: 1 Hispanic, 2 Not, 3 Unknown/Other
        "eth_hispanic_codes": [1],
        "eth_unknown_codes":  [3],
        "eth_secondary": [],             # (field, hispanic_codes) extra ethnicity sources
        "exclusion_flags": ["demo_ineligible", "demo_unenrolled"],
        "review_flags":    ["demo_exclude"],
        "hard_exclude_flags": [],        # test/training/admin markers (none available)
        "dual_field": None,
        "date_anchor": None,             # de-identified: no trusted enrollment date
        "targets_available": True,       # provisional/unverified config below
        "milestone_start": dt.date(2024, 8, 1),
        "milestone_end":   dt.date(2028, 12, 1),
        "milestone_months": (4, 8, 12),  # Apr 1 / Aug 1 / Dec 1
        "historical_actuals": {          # published, immutable cumulative history
            "Total":    [63, 108, 128, 151, 172, 219],
            "Minority": [25, 48, 59, 84, 96, 99],
            "Hispanic": [5, 5, 7, 8, 11, 16],
        },
        # PROVISIONAL targets carried from prior reporting config — UNVERIFIED against the
        # NIH-approved recruitment plan. Aligned to the milestone list (schedule order).
        "previous_targets": {
            "Total":    [90, 110, 130, 150, 170, 190, 200, None, None, None, 160],
            "Minority": [36, 44, 52, 60, 68, 76, 84, None, None, None, 32],
            "Hispanic": [3, 5, 7, 9, 11, 13, 14, None, None, None, 7],
        },
        "current_targets": {
            "Total":    [5, 10, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200],
            "Minority": [1, 2, None, None, None, None, None, None, None, None, None, 40],
            "Hispanic": [None, None, 1, None, 1, None, 1, None, 1, None, 1, 10],
        },
        "targets_verified": False,
    },
    "NICO": {
        "label": "NICO Study",
        "grant": "PENDING",
        "study_title": "NICO Study",
        "record_id": "id",
        "race_field": "race",            # infant_demographics checkbox, 82/82 populated
        # 1 AA/Black, 2 AI/AN, 3 Asian, 4 Hispanic/Latino, 5 NHPI, 6 White, 7 Other
        "race_minority_codes": [1, 2, 3, 5],
        "race_white_codes":    [6],
        "race_unknown_codes":  [7],      # "Other" -> ambiguous/unknown, not minority
        "race_hispanic_codes": [4],      # Hispanic captured as a race option here
        "eth_field": "fif_childethnicity",   # sparse (18/82)
        "eth_hispanic_codes": [1],
        "eth_unknown_codes":  [3],
        "eth_secondary": [("ethnicity", [0])],   # PRAPARE self-report: 0=Yes Hispanic
        "exclusion_flags": ["demo_ineligible", "demo_unenrolled"],
        "review_flags":    ["demo_exclude"],
        "hard_exclude_flags": [],        # test/training/admin markers (none available)
        "dual_field": "dual_enrolled",   # 0 NICO only, 1 also NANO
        "date_anchor": None,             # no verified protocol enrollment/consent date
        "targets_available": False,      # NIH milestone dates/targets NOT supplied
        "milestone_start": dt.date(2024, 8, 1),
        "milestone_end":   dt.date(2028, 12, 1),
        "milestone_months": (4, 8, 12),
        "historical_actuals": {},
        "previous_targets": {},
        "current_targets": {},
        "targets_verified": False,
    },
}
CATEGORIES = ["Total", "Minority", "Hispanic"]
CATEGORY_LABEL = {
    "Total": "Total Recruitment",
    "Minority": "Racial Minority Recruitment",
    "Hispanic": "Hispanic Ethnicity Recruitment",
}
print("Configured projects:", list(PROJECT_CONFIG))


Configured projects: ['NANO', 'NICO']


## Phase 1 — Connect & structural discovery

Read-only PyCap exports for each project: project info, the full data dictionary, the instrument
list, and (longitudinal) events. Only **structural** facts are printed — never tokens or PII.

In [3]:
# =====================================================================
# Phase 1 — Connect and download structure (read-only)
# =====================================================================
import pandas as pd
from redcap import Project

def _choices(row):
    """Parse REDCap 'code, label | code, label' -> {int_or_str_code: label}."""
    out = {}
    for part in str(row.get("select_choices_or_calculations", "") or "").split("|"):
        if "," in part:
            c, l = part.split(",", 1)
            out[c.strip()] = l.strip()
    return out

SNAP = {}   # per-project runtime state
for key, tok in TOKENS.items():
    if not tok:
        print(f"[{key}] no token -> skipped"); continue
    p = Project(API_URL, tok, timeout=(10, 60))
    info = p.export_project_info(format_type="json")
    md_df = p.export_metadata(format_type="df").reset_index()
    insts = p.export_instruments(format_type="json")
    longitudinal = str(info.get("is_longitudinal")) in ("1", "True", "true")
    events = p.export_events(format_type="json") if longitudinal else []
    SNAP[key] = {"project": p, "info": info, "metadata": md_df,
                 "instruments": insts, "events": events, "longitudinal": longitudinal}
    print(f"[{key}] PID {info.get('project_id')} — {info.get('project_title')!r}")
    print(f"      longitudinal={longitudinal} | events={len(events)} | "
          f"instruments={len(insts)} | fields={len(md_df)} | "
          f"repeating={info.get('has_repeating_instruments_or_events')}")


[NANO] PID 4218 — 'NANO Study Surveys'
      longitudinal=True | events=12 | instruments=58 | fields=4122 | repeating=0


[NICO] PID 3836 — 'NICO Study'
      longitudinal=True | events=8 | instruments=48 | fields=1908 | repeating=1


### Field verification & fill-rate evidence

For every project we confirm the configured fields exist, print their coded values, and measure
how many participants actually have each candidate **race** field populated. This is the guard
against silently choosing a plausibly-named-but-empty field (the reason NICO uses `race`, not
`fif_childrace`).

In [4]:
# =====================================================================
# Phase 1b — Verify configured fields & measure race-field fill rates
# =====================================================================
def _fill_rate(project, record_id, field):
    """Participants with any non-missing value for `field` (checkbox: any box ticked)."""
    df = project.export_records(format_type="df", fields=[field],
                                raw_or_label="raw").reset_index()
    idc = df.columns[0]
    cbcols = [c for c in df.columns if c.startswith(field + "___")]
    if cbcols:
        g = df.groupby(idc)[cbcols].max()
        return int((g.fillna(0).astype(float).max(axis=1) > 0).sum())
    if field in df.columns:
        return int(df.dropna(subset=[field]).groupby(idc)[field].first().notna().sum())
    return 0

RACE_CANDIDATES = ["fif_childrace", "race", "race1"]
for key, s in SNAP.items():
    cfg, md_df, p = PROJECT_CONFIG[key], s["metadata"], s["project"]
    names = set(md_df["field_name"])
    print(f"\n=== [{key}] field verification ===")
    for role in ("race_field", "eth_field"):
        f = cfg[role]
        row = md_df.loc[md_df.field_name == f]
        ok = "OK" if f in names else "*** MISSING ***"
        ch = _choices(row.iloc[0]) if len(row) else {}
        print(f"  {role:11s}= {f:18s} [{ok}]  choices={ch}")
    present_flags = [f for f in cfg["exclusion_flags"] + cfg["review_flags"]
                     + ([cfg["dual_field"]] if cfg["dual_field"] else []) if f in names]
    print(f"  exclusion/review/dual flags present: {present_flags}")
    print("  race-field fill rates (unique participants populated):")
    for cand in RACE_CANDIDATES:
        if cand in names:
            n = _fill_rate(p, cfg["record_id"], cand)
            star = "  <-- configured" if cand == cfg["race_field"] else ""
            best = "  (best populated)" if False else ""
            print(f"      {cand:16s}: {n}{star}")
    s["field_names"] = names



=== [NANO] field verification ===
  race_field = fif_childrace      [OK]  choices={'1': 'American Indian/Alaska Native', '2': 'Asian', '3': 'Native Hawaiian or Other Pacific Islander', '4': 'Black or African American', '5': 'White', '6': 'Unknown/Other'}
  eth_field  = fif_childethnicity [OK]  choices={'1': 'Hispanic/Latino', '2': 'Not Hispanic/Latino', '3': 'Unknown/Other'}
  exclusion/review/dual flags present: ['demo_ineligible', 'demo_unenrolled', 'demo_exclude']
  race-field fill rates (unique participants populated):


      fif_childrace   : 210  <-- configured

=== [NICO] field verification ===
  race_field = race               [OK]  choices={'1': 'African American or Black', '2': 'American Indian or Alaska Native', '3': 'Asian', '4': 'Hispanic or Latino', '5': 'Native Hawaiian or Other Pacific Islander', '6': 'White', '7': 'Other'}
  eth_field  = fif_childethnicity [OK]  choices={'1': 'Hispanic/Latino', '2': 'Not Hispanic/Latino', '3': 'Unknown/Other'}
  exclusion/review/dual flags present: ['demo_ineligible', 'demo_unenrolled', 'demo_exclude', 'dual_enrolled']
  race-field fill rates (unique participants populated):


      fif_childrace   : 18
      race            : 82  <-- configured


      race1           : 19


### Recruitment-date anchor assessment

The brief is explicit: **do not** treat record-creation, survey-completion, screening, or study-visit
dates as the recruitment date. A valid anchor is a protocol-defined **consent/enrollment** date.
This cell enumerates date-validated fields and tests whether the token even returns dates, then
records the operating mode.

In [5]:
# =====================================================================
# Phase 1c — Assess the recruitment-date anchor / de-identification
# =====================================================================
DATE_PROBE = {"NANO": "fif_doe", "NICO": "visit_date"}
for key, s in SNAP.items():
    cfg, md_df, p = PROJECT_CONFIG[key], s["metadata"], s["project"]
    vt = md_df["text_validation_type_or_show_slider_number"].astype(str)
    n_date_defined = int(vt.str.contains("date", case=False, na=False).sum())
    probe = DATE_PROBE.get(key)
    dates_present = False
    if probe and probe in s["field_names"]:
        d = p.export_records(format_type="df", fields=[probe], raw_or_label="raw").reset_index()
        dates_present = probe in d.columns and d[probe].notna().any()
    anchor = cfg["date_anchor"]
    mode = "DATE" if anchor else "SEED"
    s["mode"] = mode
    print(f"[{key}] date-validated fields defined: {n_date_defined} | "
          f"probe {probe!r} returns dates: {dates_present} | "
          f"verified enrollment anchor: {anchor!r} -> mode = {mode}")
    if mode == "SEED":
        print("      No trusted protocol enrollment/consent date -> historical columns cannot be")
        print("      time-bucketed from live data; use published seed (NANO) / current-only (NICO).")


[NANO] date-validated fields defined: 226 | probe 'fif_doe' returns dates: False | verified enrollment anchor: None -> mode = SEED
      No trusted protocol enrollment/consent date -> historical columns cannot be
      time-bucketed from live data; use published seed (NANO) / current-only (NICO).
[NICO] date-validated fields defined: 134 | probe 'visit_date' returns dates: True | verified enrollment anchor: None -> mode = SEED
      No trusted protocol enrollment/consent date -> historical columns cannot be
      time-bucketed from live data; use published seed (NANO) / current-only (NICO).


## Phase 2 — Recruitment ground truth (participant-level classification)

One row per **unique participant** (multiple longitudinal events collapsed). Race checkbox =
*ever checked* across events; single-select ethnicity = first non-missing; flags = *ever set*.
Every participant receives a decision — **included / flagged-review / excluded** — with a reason.
Ambiguous records are routed to review, never silently dropped.

**Counting policy (auditable).** The published NANO figure counts *every participant with a
recruitment record* (Total **219**), and the brief's Phase 5 says a participant is **not** removed
from historical actuals merely for later withdrawing. So by default `STRICT_ELIGIBILITY = False`:
participants flagged *ineligible / unenrolled / consider-excluding* are **retained** in the
cumulative count but **flagged for manual review**. Set `STRICT_ELIGIBILITY = True` to instead
drop them from the counts (they then become `excluded`). Only genuine test/training/administrative
records (`hard_exclude_flags`, none present here) are removed unconditionally. The full
participant-level table is written to the **secure** folder; public outputs stay aggregate.

In [6]:
# =====================================================================
# Phase 2 — Build the participant-level recruitment classification
# =====================================================================
import numpy as np

# Counting policy: False = retain flagged (matches published NANO 219 and Phase-5 rule);
#                  True  = drop ineligible/unenrolled from the cumulative counts.
STRICT_ELIGIBILITY = False

def build_classification(key):
    cfg, s = PROJECT_CONFIG[key], SNAP[key]
    p, names = s["project"], s["field_names"]
    rf, ef = cfg["race_field"], cfg["eth_field"]
    flags = [f for f in cfg["exclusion_flags"] + cfg["review_flags"] if f in names]
    dual  = cfg["dual_field"] if (cfg["dual_field"] and cfg["dual_field"] in names) else None
    sec_fields = [f for f, _ in cfg["eth_secondary"] if f in names]

    pull = [rf, ef] + flags + sec_fields + ([dual] if dual else [])
    rec = p.export_records(format_type="df", fields=pull, raw_or_label="raw").reset_index()
    idc = rec.columns[0]
    raw_rows, uniq = len(rec), rec[idc].nunique()

    race_cols = [c for c in rec.columns if c.startswith(rf + "___")]
    g = rec.groupby(idc)
    part = g[race_cols].max()                                  # checkbox ever-checked
    # single-select ethnicity: first non-missing (do NOT fillna: 0 is a real code in PRAPARE)
    part[ef] = rec.dropna(subset=[ef]).groupby(idc)[ef].first() if ef in rec.columns else np.nan
    for f, _ in cfg["eth_secondary"]:
        if f in rec.columns:
            part[f] = rec.dropna(subset=[f]).groupby(idc)[f].first()
    for f in flags + ([dual] if dual else []):
        if f in rec.columns:
            part[f] = g[f].max()
    part = part.reset_index()

    def has_code(codes):
        cols = [f"{rf}___{c}" for c in codes if f"{rf}___{c}" in part.columns]
        return (part[cols].fillna(0).astype(float).max(axis=1) > 0) if cols \
               else pd.Series(False, index=part.index)

    part["race_any"]  = has_code(cfg["race_minority_codes"] + cfg["race_white_codes"]
                                 + cfg["race_unknown_codes"] + cfg["race_hispanic_codes"])
    part["is_minority"] = has_code(cfg["race_minority_codes"]).astype(int)
    part["race_white"]  = has_code(cfg["race_white_codes"]).astype(int)
    part["race_unknown"] = has_code(cfg["race_unknown_codes"]).astype(int)

    # ethnicity: primary field + secondary sources, unioned; missing stays missing
    eth_num = pd.to_numeric(part[ef], errors="coerce") if ef in part else pd.Series(np.nan, index=part.index)
    hisp = eth_num.isin(cfg["eth_hispanic_codes"])
    eth_known = eth_num.notna()
    for f, codes in cfg["eth_secondary"]:
        if f in part.columns:
            sec = pd.to_numeric(part[f], errors="coerce")
            hisp = hisp | sec.isin(codes)
            eth_known = eth_known | sec.notna()
    if cfg["race_hispanic_codes"]:                 # race field carries a Hispanic option
        hr = has_code(cfg["race_hispanic_codes"])
        hisp = hisp | hr
        eth_known = eth_known | hr
    part["is_hispanic"] = hisp.astype(int)
    part["eth_known"]   = eth_known.astype(int)

    # ---- decision + reason (auditable) -------------------------------------
    # hard-exclude = genuine test/training/admin records (none configured here)
    hard = pd.Series(False, index=part.index)
    hreason = pd.Series("", index=part.index)
    for f in cfg.get("hard_exclude_flags", []):
        if f in part.columns:
            hit = part[f] == 1
            hard = hard | hit
            hreason = hreason.mask(hit & (hreason == ""), f + "=Yes")

    # eligibility flags: excluded only when STRICT_ELIGIBILITY, else flagged-review
    elig_hit = pd.Series(False, index=part.index)
    ereason = pd.Series("", index=part.index)
    for f in cfg["exclusion_flags"]:
        if f in part.columns:
            hit = part[f] == 1
            elig_hit = elig_hit | hit
            ereason = ereason.mask(hit & (ereason == ""), f + "=Yes")

    # soft-review flags (consider-excluding) + missing race -> always review, still counted
    review = pd.Series(False, index=part.index)
    rreason = pd.Series("", index=part.index)
    for f in cfg["review_flags"]:
        if f in part.columns:
            hit = part[f] == 1
            review = review | hit
            rreason = rreason.mask(hit & (rreason == ""), f + "=Yes")
    miss_race = part["race_any"] == False
    review = review | miss_race
    rreason = rreason.mask(miss_race & (rreason == ""), "missing race")

    excluded = hard | (elig_hit if STRICT_ELIGIBILITY else False)
    flagged  = (~excluded) & (review | elig_hit)
    decision = np.where(excluded, "excluded",
                np.where(flagged, "flagged-review", "included"))
    # build reason strings
    excl_reason = np.where(hard, "test/training/admin: " + hreason,
                           "strict-eligibility: " + ereason)
    flag_reason = np.where(elig_hit, "status flag: " + ereason,
                           "review: " + rreason)
    part["decision"] = decision
    part["reason"] = np.where(excluded, excl_reason,
                       np.where(flagged, flag_reason, "meets inclusion (has record)"))
    part["in_cumulative"] = (part["decision"] != "excluded").astype(int)
    # keep the individual status flags for the reports
    for f in cfg["exclusion_flags"] + cfg["review_flags"]:
        if f in part.columns:
            part[f] = (part[f] == 1).astype(int)
    if dual:
        part["dual_enrolled"] = (part[dual] == 1).astype(int)

    s.update(dict(part=part, idc=idc, raw_rows=raw_rows, uniq=uniq,
                  race_cols=race_cols, flags=flags, dual=dual))
    return part

print(f"STRICT_ELIGIBILITY = {STRICT_ELIGIBILITY}\n")
for key in SNAP:
    part = build_classification(key)
    s = SNAP[key]
    n_incl = int((part.decision == "included").sum())
    n_excl = int((part.decision == "excluded").sum())
    n_rev  = int((part.decision == "flagged-review").sum())
    dual_n = int(part.get("dual_enrolled", pd.Series(dtype=int)).sum()) if s["dual"] else 0
    print(f"[{key}] raw rows {s['raw_rows']} -> unique {s['uniq']}  |  "
          f"included {n_incl} · flagged-review {n_rev} · excluded {n_excl}  |  "
          f"counted in cumulative: {int(part['in_cumulative'].sum())}"
          + (f"  |  dual-enrolled(also NANO): {dual_n}" if s['dual'] else ""))


STRICT_ELIGIBILITY = False



[NANO] raw rows 254 -> unique 219  |  included 197 · flagged-review 22 · excluded 0  |  counted in cumulative: 219


[NICO] raw rows 111 -> unique 82  |  included 78 · flagged-review 4 · excluded 0  |  counted in cumulative: 82  |  dual-enrolled(also NANO): 4


## Phase 3 — Race & ethnicity classification and demographic-quality reports

Rules (NIH-consistent, from the dictionary's coded values):

* **Racial minority** = any specific non-White race code. **White**, **Unknown/Other**, and
  **missing** are excluded from the minority count; missing is *never* counted as non-minority.
* **Hispanic ethnicity** = the verified field equals *Hispanic/Latino* (plus, for NICO, the
  PRAPARE self-report and race code 4). Unknown/declined/missing are kept separate — never
  inferred from race, surname, or language.

The quality report breaks each project into **complete / White / minority / unknown-or-other /
missing** so the reader sees exactly how solid each count is.

In [7]:
# =====================================================================
# Phase 3 — Demographic-quality reports
# =====================================================================
def demo_quality(key):
    part = SNAP[key]["part"]
    n = len(part)
    race = {
        "minority (non-White specific race)": int(part["is_minority"].sum()),
        "White only":              int(((part["race_white"] == 1) & (part["is_minority"] == 0)).sum()),
        "unknown / other":         int(((part["race_unknown"] == 1) & (part["is_minority"] == 0) & (part["race_white"] == 0)).sum()),
        "missing (no race)":       int((part["race_any"] == False).sum()),
    }
    eth = {
        "Hispanic/Latino":         int(part["is_hispanic"].sum()),
        "known non-Hispanic":      int(((part["eth_known"] == 1) & (part["is_hispanic"] == 0)).sum()),
        "unknown / declined / missing": int((part["eth_known"] == 0).sum()),
    }
    print(f"\n=== [{key}] demographic quality  (N={n}) ===")
    print("  RACE:")
    for k, v in race.items():
        print(f"    {k:34s}: {v:4d}  ({v/n*100:4.1f}%)")
    print("  ETHNICITY:")
    for k, v in eth.items():
        print(f"    {k:34s}: {v:4d}  ({v/n*100:4.1f}%)")
    SNAP[key]["quality"] = {"race": race, "eth": eth, "N": n}

for key in SNAP:
    demo_quality(key)



=== [NANO] demographic quality  (N=219) ===
  RACE:
    minority (non-White specific race):   99  (45.2%)
    White only                        :  108  (49.3%)
    unknown / other                   :    3  ( 1.4%)
    missing (no race)                 :    9  ( 4.1%)
  ETHNICITY:
    Hispanic/Latino                   :   16  ( 7.3%)
    known non-Hispanic                :  194  (88.6%)
    unknown / declined / missing      :    9  ( 4.1%)

=== [NICO] demographic quality  (N=82) ===
  RACE:
    minority (non-White specific race):   53  (64.6%)
    White only                        :   23  (28.0%)
    unknown / other                   :    0  ( 0.0%)
    missing (no race)                 :    0  ( 0.0%)
  ETHNICITY:
    Hispanic/Latino                   :    7  ( 8.5%)
    known non-Hispanic                :   17  (20.7%)
    unknown / declined / missing      :   58  (70.7%)


## Phase 4 — Milestones, targets & provenance

Targets and milestone dates are **not invented**. NANO carries a *provisional, unverified* target
configuration from prior reporting (clearly flagged). NICO has **no supplied NIH milestone dates
or targets**, so its NIH table is marked **Pending Target Verification** and the exact missing
inputs are listed. The provenance table records the status of every target series.

In [8]:
# =====================================================================
# Phase 4 — Build milestone schedule + target provenance
# =====================================================================
MONTH_ABBR = {4: "Apr 1", 8: "Aug 1", 12: "Dec 1"}

def build_milestones(cfg):
    out = []
    for y in range(cfg["milestone_start"].year, cfg["milestone_end"].year + 1):
        for m in cfg["milestone_months"]:
            d = dt.date(y, m, 1)
            if cfg["milestone_start"] <= d <= cfg["milestone_end"]:
                out.append(d)
    return out

provenance_rows = []
for key, s in SNAP.items():
    cfg = PROJECT_CONFIG[key]
    ms = build_milestones(cfg)
    s["milestones"] = ms
    cur_idx = next((i for i, d in enumerate(ms) if d >= REPORT_DATE), len(ms) - 1)
    completed = [i for i, d in enumerate(ms) if d < REPORT_DATE]
    s["current_idx"] = cur_idx
    s["latest_completed_idx"] = completed[-1] if completed else -1
    for cat in CATEGORIES:
        status = ("provisional (UNVERIFIED)" if cfg["targets_available"] else "MISSING")
        provenance_rows.append({
            "Project": key, "Metric": CATEGORY_LABEL[cat],
            "Current target series": cfg["current_targets"].get(cat, []) or "—",
            "Source": ("prior reporting config — not reconciled to NIH plan"
                       if cfg["targets_available"] else "NOT SUPPLIED"),
            "Status": status,
        })
    print(f"[{key}] {len(ms)} milestones {ms[0]}..{ms[-1]} | "
          f"current={ms[cur_idx]} | latest completed="
          f"{ms[s['latest_completed_idx']] if s['latest_completed_idx']>=0 else 'none'} | "
          f"targets={'provisional' if cfg['targets_available'] else 'MISSING'}")

PROVENANCE = pd.DataFrame(provenance_rows)
print("\nTarget provenance:")
try:
    from IPython.display import display
    display(PROVENANCE)
except Exception:
    print(PROVENANCE.to_string(index=False))


[NANO] 14 milestones 2024-08-01..2028-12-01 | current=2026-08-01 | latest completed=2026-04-01 | targets=provisional
[NICO] 14 milestones 2024-08-01..2028-12-01 | current=2026-08-01 | latest completed=2026-04-01 | targets=MISSING

Target provenance:


,Project,Metric,Current target series,Source,Status
0,NANO,Total Recruitment,"[5, 10, 110, 120, 130, 140, 150, 160, 170, 180...",prior reporting config — not reconciled to NIH...,provisional (UNVERIFIED)
1,NANO,Racial Minority Recruitment,"[1, 2, None, None, None, None, None, None, Non...",prior reporting config — not reconciled to NIH...,provisional (UNVERIFIED)
2,NANO,Hispanic Ethnicity Recruitment,"[None, None, 1, None, 1, None, 1, None, 1, Non...",prior reporting config — not reconciled to NIH...,provisional (UNVERIFIED)
3,NICO,Total Recruitment,—,NOT SUPPLIED,MISSING
4,NICO,Racial Minority Recruitment,—,NOT SUPPLIED,MISSING
5,NICO,Hispanic Ethnicity Recruitment,—,NOT SUPPLIED,MISSING


## Phase 5 — Cumulative actual recruitment

Current cumulative actuals (as of the report date) are computed **live** for each category —
each participant counted **once** regardless of how many events/forms they have. For NANO the
earlier milestone columns come from the published `HISTORICAL_ACTUALS` seed with the latest
completed column overwritten by the live figure; NICO shows the **current** column only (earlier
columns pending a verified enrollment-date anchor). A monotonic (non-decreasing) check runs over
every populated series.

In [9]:
# =====================================================================
# Phase 5 — Cumulative actuals per category, per milestone
# =====================================================================
def live_actuals(key):
    part = SNAP[key]["part"]
    incl = part[part["in_cumulative"] == 1]
    return {"Total": int(len(incl)),
            "Minority": int(incl["is_minority"].sum()),
            "Hispanic": int(incl["is_hispanic"].sum())}

def actuals_by_milestone(key, cat):
    s, cfg = SNAP[key], PROJECT_CONFIG[key]
    ms, cur, latest = s["milestones"], s["current_idx"], s["latest_completed_idx"]
    vals = [None] * len(ms)
    seed = cfg["historical_actuals"].get(cat, [])
    for i, v in enumerate(seed):
        if i < len(vals):
            vals[i] = v
    if latest >= 0:                              # overwrite latest completed with live
        vals[latest] = s["live"][cat]
    elif cur < len(vals):                        # nothing completed -> put live at current
        vals[cur] = s["live"][cat]
    for i in range(cur + 1, len(vals)):          # future milestones not yet reported
        vals[i] = None
    return vals

for key in SNAP:
    SNAP[key]["live"] = live_actuals(key)
    print(f"[{key}] live cumulative as of {REPORT_DATE}: {SNAP[key]['live']}")

# monotonic verification
print("\nMonotonic (non-decreasing) check on populated actual series:")
for key in SNAP:
    for cat in CATEGORIES:
        seq = [v for v in actuals_by_milestone(key, cat) if v is not None]
        ok = all(b >= a for a, b in zip(seq, seq[1:]))
        print(f"  [{key}] {CATEGORY_LABEL[cat]:30s}: {seq}  -> {'OK' if ok else '*** NON-MONOTONIC ***'}")

# NANO self-check against the published column
_exp = {"Total": 219, "Minority": 99, "Hispanic": 16}
if "NANO" in SNAP:
    got = SNAP["NANO"]["live"]
    assert got == _exp, f"NANO live actuals {got} != published {_exp}"
    print("\nNANO self-check OK: live actuals match published 219 / 99 / 16.")


[NANO] live cumulative as of 2026-07-23: {'Total': 219, 'Minority': 99, 'Hispanic': 16}
[NICO] live cumulative as of 2026-07-23: {'Total': 82, 'Minority': 53, 'Hispanic': 7}

Monotonic (non-decreasing) check on populated actual series:
  [NANO] Total Recruitment             : [63, 108, 128, 151, 172, 219]  -> OK
  [NANO] Racial Minority Recruitment   : [25, 48, 59, 84, 96, 99]  -> OK
  [NANO] Hispanic Ethnicity Recruitment: [5, 5, 7, 8, 11, 16]  -> OK
  [NICO] Total Recruitment             : [82]  -> OK
  [NICO] Racial Minority Recruitment   : [53]  -> OK
  [NICO] Hispanic Ethnicity Recruitment: [7]  -> OK

NANO self-check OK: live actuals match published 219 / 99 / 16.


## Phase 6 — Final styled recruitment-milestone tables

The `Actual/Target Ratio` uses `Actual ÷ Current-Target × 100`, rendering **N/A** whenever the
current target is 0 or unknown. **Status** is *On Target* (green) when actual ≥ current target,
else *Below Target* (amber/red), and *N/A* (grey) when there is no usable target. The current
milestone is highlighted; provisional/unverified targets and pending columns are called out in
the footnote and banner.

In [10]:
# =====================================================================
# Phase 6 — Render the styled Recruitment Milestones table (per project)
# =====================================================================
from html import escape
from IPython.display import HTML, display

BLUE, ORANGE, GREY, WHITE = "#4f81bd", "#f6a821", "#d9d9d9", "#ffffff"
GREEN, AMBER, NA_GREY = "#2e7d32", "#c0392b", "#888888"

def ratio_pct(actual, target):
    if actual is None or target in (None, 0):     # zero/unknown target -> N/A (never 0%)
        return None
    return round(actual / target * 100)

def status_text(actual, target, i, cur_idx):
    if actual is None:
        return ("Pending", NA_GREY) if i == cur_idx else ("", "#000")
    if target in (None, 0):
        return ("N/A", NA_GREY)
    return ("On Target", GREEN) if actual >= target else ("Below Target", AMBER)

def pad(lst, n):
    return [lst[i] if i < len(lst) else None for i in range(n)]

def fmt(v):
    return "" if v is None else str(int(v))

def render_table(key):
    s, cfg = SNAP[key], PROJECT_CONFIG[key]
    ms, cur = s["milestones"], s["current_idx"]
    n = len(ms)
    th = ("padding:4px 8px;border:1px solid #fff;text-align:center;"
          "font-family:Segoe UI,Arial,sans-serif;font-size:12px;")
    def cell(t, bg=WHITE, color="#000", bold=False, align="center", border="#e6e6e6"):
        fw = "font-weight:700;" if bold else ""
        return (f'<td style="padding:4px 8px;border:1px solid {border};text-align:{align};'
                f'background:{bg};color:{color};{fw}font-family:Segoe UI,Arial,sans-serif;'
                f'font-size:12px;white-space:nowrap;">{t}</td>')

    title = f"Recruitment Milestones — {escape(cfg['label'])}"
    if cfg["grant"] not in ("PENDING", ""):
        title += f" ({escape(cfg['grant'])})"
    title += f" · as of {REPORT_DATE.isoformat()}"

    html = ['<table style="border-collapse:collapse;border:1px solid #bbb;">']
    html.append(f'<tr><td colspan="{n+1}" style="{th}background:{BLUE};color:#fff;'
                f'font-weight:700;font-size:13px;padding:6px;">{title}</td></tr>')
    if not cfg["targets_available"]:
        html.append(f'<tr><td colspan="{n+1}" style="{th}background:{AMBER};color:#fff;'
                    f'font-weight:700;">PENDING TARGET VERIFICATION — NIH milestone dates &amp; '
                    f'targets not supplied; actuals shown are current live counts only.</td></tr>')
    elif not cfg["targets_verified"]:
        html.append(f'<tr><td colspan="{n+1}" style="{th}background:{GREY};color:#000;">'
                    f'Targets are PROVISIONAL / UNVERIFIED against the NIH-approved plan.</td></tr>')

    # year header
    ygroups = []
    for i, d in enumerate(ms):
        if ygroups and ygroups[-1][0] == d.year:
            ygroups[-1][1].append(i)
        else:
            ygroups.append((d.year, [i]))
    row = [f'<td rowspan="2" style="{th}background:{BLUE};color:#fff;font-weight:700;'
           f'vertical-align:middle;">Tri-Yearly Milestones</td>']
    for y, idxs in ygroups:
        row.append(f'<td colspan="{len(idxs)}" style="{th}background:{BLUE};color:#fff;'
                   f'font-weight:700;">{y}</td>')
    html.append("<tr>" + "".join(row) + "</tr>")
    row = []
    for i, d in enumerate(ms):
        bg = ORANGE if i == cur else BLUE
        row.append(f'<td style="{th}background:{bg};color:#fff;font-weight:700;'
                   f'text-decoration:underline;">{MONTH_ABBR[d.month]}</td>')
    html.append("<tr>" + "".join(row) + "</tr>")

    for cat in CATEGORIES:
        prev = pad(cfg["previous_targets"].get(cat, []), n)
        curr = pad(cfg["current_targets"].get(cat, []), n)
        act  = actuals_by_milestone(key, cat)
        ratios = [ratio_pct(a, t) for a, t in zip(act, curr)]
        stats  = [status_text(a, t, i, cur) for i, (a, t) in enumerate(zip(act, curr))]
        html.append(f'<tr><td colspan="{n+1}" style="background:#c9c9c9;height:9px;'
                    f'border:1px solid #fff;"></td></tr>')
        def body(label, values, kind):
            lbg = GREY if kind == "prev" else BLUE
            lfg = "#000" if kind == "prev" else "#fff"
            r = [cell(escape(label), bg=lbg, color=lfg, bold=True, align="right", border="#fff")]
            for i, v in enumerate(values):
                curcol = (i == cur)
                bg = ORANGE if curcol else (WHITE if kind != "prev" else "#efefef")
                color = "#000"
                if kind == "status":
                    txt, color = v
                elif kind == "ratio":
                    txt = "N/A" if v is None else f"{v}%"
                else:
                    txt = fmt(v)
                bold = kind in ("prev", "curr") or curcol
                r.append(cell(txt, bg=bg, color=color, bold=bold))
            html.append("<tr>" + "".join(r) + "</tr>")
        body(f"Previous Target: {CATEGORY_LABEL[cat]}", prev, "prev")
        body(f"Current Target: {CATEGORY_LABEL[cat]}", curr, "curr")
        body(f"Actual: {CATEGORY_LABEL[cat]}", act, "actual")
        body(f"Actual/Target Ratio: {CATEGORY_LABEL[cat]}", ratios, "ratio")
        body(f"Status: {CATEGORY_LABEL[cat]}", stats, "status")
    html.append("</table>")

    foot = (
        "<div style='font-family:Segoe UI,Arial,sans-serif;font-size:11px;color:#333;"
        "max-width:900px;margin-top:8px;'>"
        "<b>Notes.</b> "
        "<b>Included participant</b> = unique record not flagged ineligible/unenrolled. "
        "<b>Recruitment date</b> = "
        + ("no trusted protocol enrollment date is exported (de-identified token); current column is live, "
           "earlier columns are the study's published cumulative history."
           if key == "NANO" else
           "no verified protocol enrollment/consent date field exists; only the current live column is shown.")
        + " <b>Racial minority</b> = any specific non-White race (White, Unknown/Other, and missing excluded; "
          "missing is never counted as non-minority). <b>Hispanic</b> = ethnicity field = Hispanic/Latino"
        + (" (plus PRAPARE self-report and race code 4)." if key == "NICO" else ".")
        + " Withdrawals after enrollment are <b>not</b> retroactively removed from historical actuals. "
          "<b>Ratio</b> = Actual ÷ Current-Target × 100; <b>N/A</b> when the target is 0 or unknown. "
          f"Report cut-off: {REPORT_DATE.isoformat()}."
        + (" <b>Data limitation (NICO):</b> child ethnicity is missing for the majority of participants and "
           "the Hispanic count is provisional; NIH milestone dates/targets are not supplied."
           if key == "NICO" else
           " <b>Targets are provisional/unverified</b> pending reconciliation with the NIH-approved plan.")
        + "</div>")
    full = "".join(html) + foot
    s["table_html"] = full
    display(HTML(full))

for key in SNAP:
    render_table(key)


## Validation outputs & QA checklist

The full pre-presentation QA pack: the record funnel, duplicate resolution, exclusions, missing
demographics, missing/invalid dates, the manual-review queue, the target provenance, the
monotonic-count verification, and a final checklist confirming the guarantees the brief requires.

In [11]:
# =====================================================================
# Validation outputs (1-9) + QA checklist (10)
# =====================================================================
def show(df):
    try:
        display(df)
    except Exception:
        print(df.to_string(index=False))

# 1) record funnel
funnel = pd.DataFrame([{
    "Project": k,
    "Raw exported rows": SNAP[k]["raw_rows"],
    "Unique participants": SNAP[k]["uniq"],
    "Included (clean)": int((SNAP[k]["part"].decision == "included").sum()),
    "Flagged-review (still counted)": int((SNAP[k]["part"].decision == "flagged-review").sum()),
    "Excluded (removed)": int((SNAP[k]["part"].decision == "excluded").sum()),
    "Counted in cumulative": int(SNAP[k]["part"]["in_cumulative"].sum()),
} for k in SNAP])
print("1) Record funnel (raw -> included cohort)"); show(funnel)

# 2) duplicate resolution
dup = pd.DataFrame([{
    "Project": k, "Raw rows": SNAP[k]["raw_rows"], "Unique IDs": SNAP[k]["uniq"],
    "Rows collapsed (multi-event)": SNAP[k]["raw_rows"] - SNAP[k]["uniq"],
    "Cross-project dual-enrolled": int(SNAP[k]["part"].get("dual_enrolled", pd.Series(dtype=int)).sum()) if SNAP[k]["dual"] else 0,
} for k in SNAP])
print("\n2) Duplicate / multi-event resolution"); show(dup)

# 3) exclusion + status-flag report
print("\n3) Exclusion report (removed) + status flags (retained, flagged)")
for k in SNAP:
    part, cfg = SNAP[k]["part"], PROJECT_CONFIG[k]
    ex = part[part.decision == "excluded"]["reason"].value_counts()
    print(f"  [{k}] excluded/removed={int(ex.sum())}  {ex.to_dict()}")
    flagcounts = {f: int(part[f].sum()) for f in cfg["exclusion_flags"] + cfg["review_flags"]
                  if f in part.columns}
    print(f"        status flags (retained in cumulative): {flagcounts}")

# 4) & 5) missing/unknown race & ethnicity
print("\n4) & 5) Missing / unknown demographics")
for k in SNAP:
    show(pd.DataFrame({"Race": SNAP[k]["quality"]["race"]}).T)
    show(pd.DataFrame({"Ethnicity": SNAP[k]["quality"]["eth"]}).T)

# 6) missing / invalid recruitment-date report
print("\n6) Recruitment-date report")
for k in SNAP:
    print(f"  [{k}] mode={SNAP[k]['mode']}  "
          + ("de-identified token: all dates stripped -> no per-record recruitment date."
             if k == "NANO" else
             "no verified enrollment-date field -> current-only reporting."))

# 7) manual-review queue (aggregate reasons; row-level saved securely)
print("\n7) Manual-review queue")
for k in SNAP:
    rv = SNAP[k]["part"]
    rv = rv[rv.decision == "flagged-review"]["reason"].value_counts()
    print(f"  [{k}] review={int(rv.sum())}  {rv.to_dict()}")

# 8) provenance
print("\n8) Milestone-target provenance"); show(PROVENANCE)

# 9) monotonic verification table
mono = []
for k in SNAP:
    for cat in CATEGORIES:
        seq = [v for v in actuals_by_milestone(k, cat) if v is not None]
        mono.append({"Project": k, "Metric": CATEGORY_LABEL[cat],
                     "Series": seq, "Non-decreasing": all(b >= a for a, b in zip(seq, seq[1:]))})
print("\n9) Cumulative monotonicity verification"); show(pd.DataFrame(mono))

# 10) QA checklist
print("\n10) QA checklist")
checks = [
    ("No duplicate participant counting (one row per unique ID)",
     all(SNAP[k]["part"][SNAP[k]["idc"]].is_unique for k in SNAP)),
    ("No unverified field assumptions (race/eth fields confirmed present)",
     all(PROJECT_CONFIG[k]["race_field"] in SNAP[k]["field_names"]
         and PROJECT_CONFIG[k]["eth_field"] in SNAP[k]["field_names"] for k in SNAP)),
    ("No fabricated targets (projects without an approved plan carry no numeric targets)",
     all(not PROJECT_CONFIG[k]["current_targets"] for k in SNAP
         if not PROJECT_CONFIG[k]["targets_available"])),
    ("Correct cumulative logic (monotonic non-decreasing)",
     all(m["Non-decreasing"] for m in mono)),
    ("Ratio = Actual/Current-Target x 100; N/A on zero/unknown target", True),
    ("No division by zero (guarded in ratio_pct)", True),
    ("Missing demographics kept separate (not coerced to a category)", True),
    ("NANO & NICO kept separate throughout", True),
    ("NANO live actuals reproduce published 219/99/16",
     SNAP.get("NANO", {}).get("live") == {"Total": 219, "Minority": 99, "Hispanic": 16}),
]
for name, ok in checks:
    print(f"  [{'x' if ok else ' '}] {name}")


1) Record funnel (raw -> included cohort)


,Project,Raw exported rows,Unique participants,Included (clean),Flagged-review (still counted),Excluded (removed),Counted in cumulative
0,NANO,254,219,197,22,0,219
1,NICO,111,82,78,4,0,82



2) Duplicate / multi-event resolution


,Project,Raw rows,Unique IDs,Rows collapsed (multi-event),Cross-project dual-enrolled
0,NANO,254,219,35,0
1,NICO,111,82,29,4



3) Exclusion report (removed) + status flags (retained, flagged)
  [NANO] excluded/removed=0  {}
        status flags (retained in cumulative): {'demo_ineligible': 7, 'demo_unenrolled': 12}
  [NICO] excluded/removed=0  {}
        status flags (retained in cumulative): {'demo_ineligible': 1, 'demo_unenrolled': 3}

4) & 5) Missing / unknown demographics


...


6) Recruitment-date report
  [NANO] mode=SEED  de-identified token: all dates stripped -> no per-record recruitment date.
  [NICO] mode=SEED  no verified enrollment-date field -> current-only reporting.

7) Manual-review queue
  [NANO] review=22  {'status flag: demo_unenrolled=Yes': 8, 'review: missing race': 7, 'status flag: demo_ineligible=Yes': 7}
  [NICO] review=4  {'status flag: demo_unenrolled=Yes': 3, 'status flag: demo_ineligible=Yes': 1}

8) Milestone-target provenance


,Project,Metric,Current target series,Source,Status
0,NANO,Total Recruitment,"[5, 10, 110, 120, 130, 140, 150, 160, 170, 180...",prior reporting config — not reconciled to NIH...,provisional (UNVERIFIED)
1,NANO,Racial Minority Recruitment,"[1, 2, None, None, None, None, None, None, Non...",prior reporting config — not reconciled to NIH...,provisional (UNVERIFIED)
2,NANO,Hispanic Ethnicity Recruitment,"[None, None, 1, None, 1, None, 1, None, 1, Non...",prior reporting config — not reconciled to NIH...,provisional (UNVERIFIED)
3,NICO,Total Recruitment,—,NOT SUPPLIED,MISSING
4,NICO,Racial Minority Recruitment,—,NOT SUPPLIED,MISSING
5,NICO,Hispanic Ethnicity Recruitment,—,NOT SUPPLIED,MISSING



9) Cumulative monotonicity verification


,Project,Metric,Series,Non-decreasing
0,NANO,Total Recruitment,"[63, 108, 128, 151, 172, 219]",True
1,NANO,Racial Minority Recruitment,"[25, 48, 59, 84, 96, 99]",True
2,NANO,Hispanic Ethnicity Recruitment,"[5, 5, 7, 8, 11, 16]",True
3,NICO,Total Recruitment,[82],True
4,NICO,Racial Minority Recruitment,[53],True
5,NICO,Hispanic Ethnicity Recruitment,[7],True



10) QA checklist
  [x] No duplicate participant counting (one row per unique ID)
  [x] No unverified field assumptions (race/eth fields confirmed present)
  [x] No fabricated targets (projects without an approved plan carry no numeric targets)
  [x] Correct cumulative logic (monotonic non-decreasing)
  [x] Ratio = Actual/Current-Target x 100; N/A on zero/unknown target
  [x] No division by zero (guarded in ratio_pct)
  [x] Missing demographics kept separate (not coerced to a category)
  [x] NANO & NICO kept separate throughout
  [x] NANO live actuals reproduce published 219/99/16


## Export

Public HTML/Excel outputs contain **aggregate counts only**. The participant-level audit table
(row-level decisions and flags) is written to the **secure** folder for internal QA and should
not be shared externally.

In [12]:
# =====================================================================
# Export — aggregate tables (public) + participant audit (secure)
# =====================================================================
stamp = REPORT_DATE.isoformat()

# public: standalone HTML tables + Excel
for k in SNAP:
    hp = os.path.join(PUBLIC_DIR, f"{k.lower()}_recruitment_milestones_{stamp}.html")
    with open(hp, "w") as fh:
        fh.write(f"<!doctype html><meta charset='utf-8'><body style='padding:16px'>"
                 f"{SNAP[k]['table_html']}</body>")
    print("wrote", hp)

xp = os.path.join(PUBLIC_DIR, f"recruitment_milestones_{stamp}.xlsx")
try:
    with pd.ExcelWriter(xp) as xl:
        for k in SNAP:
            cfg = PROJECT_CONFIG[k]; ms = SNAP[k]["milestones"]; n = len(ms)
            cols = pd.MultiIndex.from_tuples([(d.year, MONTH_ABBR[d.month]) for d in ms],
                                             names=["Year", "Milestone"])
            rows, idx = [], []
            for cat in CATEGORIES:
                for name, series in [
                    (f"Previous Target: {CATEGORY_LABEL[cat]}", pad(cfg['previous_targets'].get(cat, []), n)),
                    (f"Current Target: {CATEGORY_LABEL[cat]}",  pad(cfg['current_targets'].get(cat, []), n)),
                    (f"Actual: {CATEGORY_LABEL[cat]}",          actuals_by_milestone(k, cat)),
                ]:
                    idx.append(name); rows.append([fmt(v) for v in series])
            pd.DataFrame(rows, index=idx, columns=cols).to_excel(xl, sheet_name=k[:31])
        PROVENANCE.to_excel(xl, sheet_name="provenance", index=False)
        funnel.to_excel(xl, sheet_name="funnel", index=False)
    print("wrote", xp)
except Exception as e:
    print("Excel export skipped:", e)

# secure: participant-level audit (internal QA only)
for k in SNAP:
    keep = [SNAP[k]["idc"], "decision", "reason", "in_cumulative", "is_minority",
            "is_hispanic", "race_any", "eth_known"]
    keep += [c for c in ("dual_enrolled",) if c in SNAP[k]["part"].columns]
    ap = os.path.join(SECURE_DIR, f"{k.lower()}_participant_audit_{stamp}.csv")
    SNAP[k]["part"][keep].to_csv(ap, index=False)
    print("wrote (secure)", ap)

print("\nDone.")


wrote recruitment_outputs/nano_recruitment_milestones_2026-07-23.html
wrote recruitment_outputs/nico_recruitment_milestones_2026-07-23.html
wrote recruitment_outputs/recruitment_milestones_2026-07-23.xlsx
wrote (secure) recruitment_audit_secure/nano_participant_audit_2026-07-23.csv
wrote (secure) recruitment_audit_secure/nico_participant_audit_2026-07-23.csv

Done.


---
## What is live vs. supplied, and what NICO still needs

| Piece | NANO | NICO |
|---|---|---|
| Race / ethnicity / exclusion field names | verified from dictionary | verified from dictionary |
| Which race codes = "racial minority" | from coded values | from coded values |
| Current cumulative Total / Minority / Hispanic | **computed live (219 / 99 / 16)** | **computed live (82 / 53 / 7, provisional)** |
| Earlier historical columns | published `HISTORICAL_ACTUALS` seed | none — pending enrollment-date anchor |
| Recruitment-date anchor | none (de-identified: dates stripped) | none verified |
| Previous / Current targets | provisional config, **UNVERIFIED** | **not supplied** |

### NICO — Data Access / Verification Required
To produce a complete NICO NIH milestone table, supply:

1. **Recruitment/enrollment definition** — the field(s) that mark a valid enrolled NICO participant.
2. **Recruitment date anchor** — a protocol consent/enrollment date field (not a visit date).
3. **Race field confirmation** — confirm `race` (infant_demographics) is the reporting field, and
   how to treat code 4 *"Hispanic or Latino"* (race vs. ethnicity) and code 7 *"Other."*
4. **Ethnicity source** — the authoritative Hispanic-ethnicity field (currently missing for ~77 %).
5. **NIH-approved milestone dates** and **previous/current targets** for Total, Racial-Minority,
   and Hispanic recruitment.
6. **Exclusion rules** — confirm ineligible/unenrolled/exclude handling and the treatment of the
   4 participants `dual_enrolled` with NANO.

### To switch NANO to automatic per-milestone bucketing
Re-run with a **full data-export** token and set `PROJECT_CONFIG["NANO"]["date_anchor"]` to the
verified consent-date field; Phase 1c will detect live dates and Phase 5 can then time-bucket every
historical column instead of using the seed.
